# N-gram recipe generator training

Install Huggingface libraries to use the pretrained tokenizer and the recipe dataset.

In [27]:
!pip install -qU git+https://github.com/huggingface/transformers.git datasets
!pip install -q googletrans==4.0.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Import modules and set random seeds.

In [28]:
import os, random, asyncio, nest_asyncio
import numpy as np
from pandas import DataFrame
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from googletrans import Translator

random.seed(20230629)
np.random.seed(20230629)
tf.random.set_seed(20230629)

plt.rcParams.update({'font.size': 10})
nest_asyncio.apply()

Download the pretrained tokenizer and check the vacabulary size.

In [29]:
from transformers import AutoTokenizer, AutoConfig
model_ckpt = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
VOCAB_SIZE = AutoConfig.from_pretrained(model_ckpt).vocab_size

print(f'Vocabulary size: {VOCAB_SIZE}')

Vocabulary size: 30522


Download the recipe dataset and extract directions texts.

In [30]:
from datasets import load_dataset
recipe = load_dataset('Shengtao/recipe')

def join_title_and_directions(title_directions):
    title, directions = title_directions
    return f'Recipe for {title}: {directions}'

recipe_texts = zip(recipe['train']['title'], recipe['train']['directions'])
recipe_texts = [*map(join_title_and_directions, recipe_texts)]
recipe_texts = recipe_texts[::4] # Select 25% of the entire training set.

In [26]:
recipe_texts[1]

"Recipe for Quick and Easy Pizza Crust: Preheat oven to 450 degrees F (230 degrees C). In a medium bowl, dissolve yeast and sugar in warm water. Let stand until creamy, about 10 minutes. Stir in flour, salt and oil. Beat until smooth. Let rest for 5 minutes. Turn dough out onto a lightly floured surface and pat or roll into a round. Transfer crust to a lightly greased pizza pan or baker's peel dusted with cornmeal. Spread with desired toppings and bake in preheated oven for 15 to 20 minutes, or until golden brown. Let baked pizza cool for 5 minutes before serving."

Create the training and test datasets, truncating long texts into 128 words.

In [7]:
MAX_LEN = 128
train_set, test_set = train_test_split(recipe_texts, test_size=0.1)

train_set = tokenizer(train_set, max_length=MAX_LEN,
                      padding='max_length', truncation=True)
train_text = np.array(train_set['input_ids'])[:, :-1]
train_label = np.array(train_set['input_ids'])[:, 1:]

test_set = tokenizer(test_set, max_length=MAX_LEN,
                     padding='max_length', truncation=True)
test_text = np.array(test_set['input_ids'])[:, :-1]
test_label = np.array(test_set['input_ids'])[:, 1:]

Define n-gram next word prediction model

In [8]:
import random
from collections import defaultdict, Counter

def build_ngram_model(texts, n=3):

    context_size = n - 1
    ngram_model = defaultdict(Counter)

    print(f"{n}-gram モデルを構築中...")

    for text in texts:
        tokens = text.split()

        if len(tokens) < n:
            continue

        for i in range(len(tokens) - context_size):
            context = tuple(tokens[i : i + context_size])
            next_word = tokens[i + context_size]

            ngram_model[context][next_word] += 1

    print(f"モデルの構築が完了しました。ユニークなコンテキスト数: {len(ngram_model)}")
    return ngram_model

Define the text generation function.

In [24]:
import time
import random
import sys

def stream_ngram_text(model, n, seed_text, max_words=50, words_per_line=12):
    tokens = seed_text.split()
    context_size = n - 1

    if len(tokens) < context_size:
        print(f"エラー: シードテキストには少なくとも {context_size} つの単語を含めてください。")
        return

    word_count = 0

    for word in tokens:
        print(word, end=' ', flush=True)
        word_count += 1
        time.sleep(0.3)

        if word_count >= words_per_line:
            print()
            word_count = 0

    result = tokens.copy()

    for _ in range(max_words):
        current_context = tuple(result[-context_size:])
        next_word_counter = model.get(current_context)

        if not next_word_counter:
            break

        words = list(next_word_counter.keys())
        weights = list(next_word_counter.values())
        next_word = random.choices(words, weights=weights, k=1)[0]

        result.append(next_word)

        print(next_word, end=' ', flush=True)
        word_count += 1
        time.sleep(0.1)

        if word_count >= words_per_line:
            print()
            word_count = 0

    print('\n\n----------\n')
    translator = Translator()
    translated = asyncio.run(translator.translate(' '.join(result), dest='ja'))
    return translated.text

Test run.

In [ ]:
N_GRAM = 4
model_4gram = build_ngram_model(recipe_texts, n=N_GRAM)

In [25]:
seed = "Recipe for Simple Macaroni"
stream_ngram_text(model_4gram, n=N_GRAM, seed_text=seed, max_words=100, words_per_line=12)

Recipe for Simple Macaroni and Cheese: Preheat the oven to 400 degrees 
F (200 degrees C). Place the eggs into big chunks and mix 
gently with your hands as possible. Add zucchini and cook until done. 
Discard bay leaf. Add vegetables, and continue cooking until the clams and 
juice. Cook, stirring occasionally, until wine has evaporated, about 1 minute. Stir 
in second cup of milk and heat just to a simmer, stirring 
often. Simmer for 5 minutes, then add coconut milk and stir into 
the skillet. Stir, scraping up browned bits and deglaze the pan. Pour 
in the evaporated milk. Season with salt to 

----------




'シンプルなマカロニ＆チーズのレシピ： オーブンを400°F（200℃）に予熱します。卵を大きめに割り入れ、手でできるだけ優しく混ぜます。ズッキーニを加えて完成するまで煮ます。月桂樹の葉は捨てます。野菜を加え、アサリと汁が出るまで煮続けます。時々かき混ぜながら、ワインが蒸発するまで約1分間煮ます。 2杯目の牛乳を加えてよくかき混ぜながら沸騰するまで加熱します。 5分ほど煮たら、ココナッツミルクを加えてフライパンに入れて混ぜます。かき混ぜながら茶色になった部分をこすり落とし、パンをディグレーズします。エバミルクを注ぎます。塩で味付けして、'